In [ ]:
import geopandas as gpd
import pyogrio
import shapely.wkb
import rasterio
from rasterio import features
from rasterio.windows import Window, bounds, transform as window_transform
from rasterio.warp import transform_bounds
from rasterio.crs import CRS
import numpy as np
import subprocess
import sqlite3
import struct
import shutil
import os
import gc
import resource
import time


# Peak resident memory of this process, in MiB (ru_maxrss is KiB on Linux).
# Kaggle OOM-kills the kernel without a Python traceback, so log this after
# every state to see which one pushes the run over the limit.
def peak_rss_mib():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


# Timing/progress log, written to stdout *and* to a file. nbconvert buffers a
# cell's stdout until the notebook finishes, so when a run is killed by the
# runtime limit the file is the only record of which state and which chunk was
# running, and of how long each stage actually took.
TIMING_LOG = "/kaggle/working/nwi_timing.log"
open(TIMING_LOG, "w").close()


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(TIMING_LOG, "a") as fh:
        fh.write(line + "\n")


def fmt(seconds):
    """Compact duration: 45s, 12.3min, 2.4h."""
    if seconds < 120:
        return f"{seconds:.0f}s"
    if seconds < 7200:
        return f"{seconds / 60:.1f}min"
    return f"{seconds / 3600:.1f}h"


def pixel_window(bounds, transform, shape):
    """Integer pixel window covering `bounds`, clamped to the raster shape."""
    left, bottom, right, top = bounds
    inv = ~transform
    col_a, row_a = inv * (left, top)
    col_b, row_b = inv * (right, bottom)

    col_0 = max(0, min(int(np.floor(min(col_a, col_b))), shape[1] - 1))
    row_0 = max(0, min(int(np.floor(min(row_a, row_b))), shape[0] - 1))
    col_1 = max(0, min(int(np.ceil(max(col_a, col_b))), shape[1]))
    row_1 = max(0, min(int(np.ceil(max(row_a, row_b))), shape[0]))

    return Window(col_0, row_0, col_1 - col_0, row_1 - row_0)


def layer_names(gpkg_path):
    """Feature layer names in the package.

    pyogrio.list_layers() opens the datasource with GDAL. That is normally
    instant, but everything else this pre-flight reads is a SQLite table, and
    on /kaggle/input an open that touches the whole file has already cost this
    job five hours - so gpkg_contents lists the same layers without it.
    """
    if gpkg_path.lower().endswith(".gpkg"):
        con = sqlite3.connect(f"file:{gpkg_path}?mode=ro", uri=True)
        try:
            return [row[0] for row in con.execute(
                "SELECT table_name FROM gpkg_contents "
                "WHERE data_type = 'features' ORDER BY table_name")]
        finally:
            con.close()
    return [layer[0] for layer in pyogrio.list_layers(gpkg_path)]


def find_wetland_layer(file_path, state_abbr):
    """Name of the layer that holds the wetland polygons.

    A state package ships several layers and their names are not uniform, so
    pick by field rather than by name: only the wetlands polygon layer has an
    ATTRIBUTE column. Attachments, <ST>_Wetlands_Historic_Map_Info, riparian
    and line layers do not, and burning one of those either fails outright or
    silently adds nothing. <ST>_Wetlands is still tried first.
    """
    names = layer_names(file_path)
    preferred = f"{state_abbr}_Wetlands"
    ordered = ([preferred] if preferred in names else []) + [
        name for name in names if name != preferred
    ]
    inspected = []
    for name in ordered:
        fields = table_fields(file_path, name)
        if "ATTRIBUTE" in fields:
            return name
        inspected.append(f"{name} {fields}")
    raise RuntimeError(
        f"{os.path.basename(file_path)}: no layer has an ATTRIBUTE column, so "
        f"there are no NWI wetland codes to filter on. Layers and fields: "
        f"{'; '.join(inspected)}. Re-download that state's wetlands polygon "
        f"package from the NWI state downloads - a '*_Historic_Map_Info' layer "
        f"is a lookup table, not the wetlands."
    )


def table_fields(gpkg_path, layer):
    """Column names of `layer`, read from the SQLite schema.

    Not pyogrio.read_info()[ "fields" ]: that call also makes GDAL answer a
    feature count and an extent, and on /kaggle/input those are whole-file
    scans. NY spent 2h48m here with nothing burned (nwi_timing.log, 17 Nov).
    PRAGMA reads the schema and nothing else.
    """
    if not gpkg_path.lower().endswith(".gpkg"):
        return list(pyogrio.read_info(gpkg_path, layer=layer)["fields"])
    con = sqlite3.connect(f"file:{gpkg_path}?mode=ro", uri=True)
    try:
        return [row[1] for row in con.execute(f'PRAGMA table_info("{layer}")')]
    finally:
        con.close()


def count_str(n):
    """Feature count, or '?' when the package carries no count table."""
    return f"{n:,}" if n is not None else "?"


def rtree_table(layer, geom_col):
    """Name of the RTree table GDAL looks for, e.g. rtree_WV_Wetlands_Shape."""
    return f"rtree_{layer}_{geom_col}"


def has_spatial_index(file_path, layer):
    """True if the GeoPackage has an RTree index on this layer's geometry.

    The burn fetches one tile at a time, so a layer with no index costs a full
    scan per tile. The table is named after the layer's geometry *column*, and
    NWI names that column "Shape", not "geom": assuming "geom" declared all
    fifteen packages unindexed and aborted two runs, one in 90s on IN and one
    after five hours on WV, when the index was there all along.
    """
    if not file_path.lower().endswith(".gpkg"):
        return True  # nothing to introspect; the burn still works, just unverified
    con = sqlite3.connect(f"file:{file_path}?mode=ro", uri=True)
    try:
        return con.execute(
            "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?",
            (rtree_table(layer, gpkg_geometry_column(file_path, layer)),),
        ).fetchone() is not None
    finally:
        con.close()


def layer_meta(gpkg_path, layer):
    """Feature count, extent and CRS, read from the package's own tables.

    Deliberately not pyogrio.read_info(): that asks GDAL for a feature count
    and an extent, which on these packages is a whole-file scan - 44s for WV's
    149k features, 2h48m for NY. Everything below is a one-row table read or an
    RTree root, so it costs milliseconds and the burn starts immediately.

    `features` is None when the package keeps no count table, `bounds` is None
    when there is no RTree to take the extent from.
    """
    con = sqlite3.connect(f"file:{gpkg_path}?mode=ro", uri=True)
    try:
        tables = {row[0] for row in con.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table'")}

        geom_col = srs_id = None
        if "gpkg_geometry_columns" in tables:
            row = con.execute(
                "SELECT column_name, srs_id FROM gpkg_geometry_columns "
                "WHERE table_name = ?",
                (layer,),
            ).fetchone()
            if row is not None:
                geom_col, srs_id = row

        # GDAL keeps this in step with triggers, so it is a single row read.
        # A package that keeps no such table reports an unknown count instead
        # of paying for a COUNT(*) over the whole file.
        features = None
        if "gpkg_ogr_contents" in tables:
            row = con.execute(
                "SELECT feature_count FROM gpkg_ogr_contents "
                "WHERE table_name = ?",
                (layer,),
            ).fetchone()
            if row is not None:
                features = int(row[0])

        # Every feature's box is already in the RTree, so its root is the extent.
        extent = None
        if geom_col is not None:
            rtree = rtree_table(layer, geom_col)
            if rtree in tables:
                row = con.execute(
                    f'SELECT MIN(minx), MIN(miny), MAX(maxx), MAX(maxy) '
                    f'FROM "{rtree}"').fetchone()
                if row is not None and row[0] is not None:
                    extent = tuple(float(v) for v in row)

        crs = None
        if srs_id is not None and "gpkg_spatial_ref_sys" in tables:
            row = con.execute(
                "SELECT organization, organization_coordsys_id, definition "
                "FROM gpkg_spatial_ref_sys WHERE srs_id = ?",
                (srs_id,),
            ).fetchone()
            if row is not None:
                organization, org_id, definition = row
                if definition and definition != "undefined":
                    try:
                        crs = CRS.from_wkt(definition)
                    except Exception:
                        crs = None
                if crs is None and organization == "EPSG" and org_id:
                    crs = CRS.from_epsg(int(org_id))
    finally:
        con.close()
    return {"features": features, "bounds": extent, "crs": crs}


def gpkg_geometry_column(gpkg_path, layer):
    """The layer's geometry column name, straight from the GeoPackage metadata."""
    con = sqlite3.connect(f"file:{gpkg_path}?mode=ro", uri=True)
    try:
        row = con.execute(
            "SELECT column_name FROM gpkg_geometry_columns WHERE table_name = ?",
            (layer,),
        ).fetchone()
    finally:
        con.close()
    if row is None:
        raise RuntimeError(
            f"{os.path.basename(gpkg_path)}: gpkg_geometry_columns has no row "
            f"for layer {layer}, so there is no geometry column to index."
        )
    return row[0]


# A GeoPackage geometry blob is an 8-byte header, then the envelope the writer
# already computed, then the WKB. For the usual XY envelope, bytes 8:40 are
# minx, maxx, miny, maxy, so the index is built from header reads instead of a
# geometry parse per feature. This maps the 4-bit envelope indicator in the
# flags byte to that envelope's size in bytes.
ENVELOPE_BYTES = {1: 32, 2: 48, 3: 48, 4: 64}


def blob_bounds(blob):
    """(minx, maxx, miny, maxy) of one geometry blob, or None if it is empty."""
    if not blob or len(blob) < 8 or blob[0:2] != b"GP":
        return None
    flags = blob[3]
    if flags & 0b00100000:              # bit 5: the geometry is empty
        return None
    order = "<" if flags & 0b00000001 else ">"        # bit 0: 1 = little endian
    envelope = ENVELOPE_BYTES.get((flags >> 1) & 0b00001111)
    if envelope is not None and len(blob) >= 8 + envelope:
        return struct.unpack(order + "4d", blob[8:40])
    # No precomputed envelope. GDAL always writes one, so this is a correctness
    # fallback: measure the WKB, which costs a shapely parse for that feature.
    geom = shapely.wkb.loads(bytes(blob[8 + (envelope or 0):]))
    if geom.is_empty:
        return None
    minx, miny, maxx, maxy = geom.bounds        # shapely's order, not the rtree's
    return (minx, maxx, miny, maxy)


def indexed_copy(file_path, layer):
    """Writable copy of `file_path` with an RTree built on `layer`; path returned.

    NWI's state downloads ship without a spatial index and /kaggle/input is
    read-only, so a per-tile bbox query has nothing to use: without an index
    every tile scans the whole layer, slower than the paging this replaced.
    Copying the database file is a raw byte copy - no re-encode, no GDAL - and
    the index itself is a header read per feature, so this costs minutes on a
    state that takes hours to burn. Peak extra disk stays at one package: the
    caller deletes the copy as soon as that state is done.
    """
    geom_col = gpkg_geometry_column(file_path, layer)
    os.makedirs(indexed_copy_dir, exist_ok=True)
    copy_path = os.path.join(indexed_copy_dir, os.path.basename(file_path))
    started = time.perf_counter()
    shutil.copyfile(file_path, copy_path)

    rtree = f"rtree_{layer}_{geom_col}"
    rows = 0
    con = sqlite3.connect(copy_path)
    try:
        # Throwaway file: durability buys nothing here and costs an fsync per batch.
        con.execute("PRAGMA journal_mode = OFF")
        con.execute("PRAGMA synchronous = OFF")
        try:
            # The table GDAL looks for when it plans a bbox query: `id` is the
            # feature's fid, the four doubles are that feature's bounding box.
            con.execute(f'CREATE VIRTUAL TABLE "{rtree}" '
                        f"USING rtree(id, minx, maxx, miny, maxy)")
        except sqlite3.OperationalError as exc:
            raise RuntimeError(
                f"{os.path.basename(file_path)}: this Python's sqlite3 cannot "
                f"create an rtree table ({exc}), so the package cannot be "
                f"indexed here. Index it with GDAL instead: "
                f"ogrinfo -sql \"SELECT CreateSpatialIndex('{layer}','{geom_col}')\" "
                f"<copy>.gpkg"
            ) from exc
        batch = []
        for fid, blob in con.execute(f'SELECT fid, "{geom_col}" FROM "{layer}"'):
            box = blob_bounds(blob)
            if box is not None:
                batch.append((fid, *box))
            if len(batch) >= 50_000:
                con.executemany(f'INSERT INTO "{rtree}" VALUES (?, ?, ?, ?, ?)',
                                batch)
                rows += len(batch)
                batch = []
        if batch:
            con.executemany(f'INSERT INTO "{rtree}" VALUES (?, ?, ?, ?, ?)', batch)
            rows += len(batch)
        # Register the index so GDAL (and ogrinfo) read it as a real one instead
        # of an orphan table. A package that uses no extension - which is what a
        # GeoPackage with no index looks like - has no gpkg_extensions table at
        # all, so create it with the spec's columns first. No sync triggers:
        # nothing ever writes this copy.
        con.execute(
            "CREATE TABLE IF NOT EXISTS gpkg_extensions ("
            "table_name TEXT, column_name TEXT, "
            "extension_name TEXT NOT NULL, definition TEXT NOT NULL, "
            "scope TEXT NOT NULL, "
            "CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))"
        )
        con.execute(
            "INSERT OR IGNORE INTO gpkg_extensions "
            "(table_name, column_name, extension_name, definition, scope) "
            "VALUES (?, ?, 'gpkg_rtree_index', "
            "'http://www.geopackage.org/spec120/#extension_rtree', 'write-only')",
            (layer, geom_col),
        )
        con.commit()
    finally:
        con.close()

    if rows == 0:
        os.remove(copy_path)
        raise RuntimeError(
            f"{os.path.basename(file_path)}: layer {layer} has no non-empty "
            f"geometry, so there is nothing to index."
        )
    log(f"    indexed {rows:,} features on a "
        f"{os.path.getsize(copy_path) / 2**20:.0f} MiB copy in "
        f"{fmt(time.perf_counter() - started)}")
    return copy_path


# 1. Configuration & Paths
nwi_folder = "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/"
master_raster_path = "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/land_cover_clipped_reproj.tiff"
temp_mask_raster = "/kaggle/working/temp_wetland_mask.tif"
output_distance_raster = "/kaggle/working/wetland_distance.tiff"

# Where indexed working copies are built (see indexed_copy). One at a time: each
# is deleted once its state has been burned, so this never holds more than the
# largest single package.
indexed_copy_dir = "/kaggle/working/nwi_indexed"

MASTER_NODATA   = 255    # Annual NLCD nodata, written by NLCD_raster_prep2.ipynb
NLCD_OPEN_WATER = 11     # NLCD class 11 -> no wetland-distance value, same as TWI

# Spatial tile size, in pixels, for the burn. Peak RAM scales with the features
# inside one tile, not with the size of a state; disk I/O scales with each
# state's footprint on the grid, read and written once. Row paging instead
# re-read and re-wrote the state's whole bounding box for every 20k polygons,
# and its skip_features walk re-scans from the first row on every page.
TILE_PX = 2048        # 2048 px * 30 m ~= 61 km on a side

# The six study states. Anything they burn stays in the output.
state_files = [
    # "IN_geopackage_wetlands.gpkg",
    # "OH_geopackage_wetlands.gpkg",
    # "MI_geopackage_wetlands.gpkg",
    # "KY_geopackage_wetlands.gpkg",
    # "PA_geopackage_wetlands.gpkg",
    "WV_geopackage_wetlands.gpkg",
]

# Neighbouring states, burned for context and then masked straight back out. Their
# wetlands are never in the output, but they are usually the nearest wetland for
# land inside the cutline: a pixel 200 m from the Indiana/Illinois line is a few
# hundred metres from an Illinois marsh, and measuring only to the nearest
# *Indiana* wetland reported kilometres instead. Ontario is the one gap that cannot
# be closed - NWI has no Canadian data - but the only Michigan-Ontario land border
# is a few km at Sault Ste. Marie, flanked by US wetlands at both ends.
# Grab these from the NWI wetland downloads (same <ST>_geopackage_wetlands.gpkg
# product) and drop them next to the study states. They roughly double the burn
# time; trim the list to IL/WI/NY if that gets in the way.
# context_files = [
#     "IL_geopackage_wetlands.gpkg",
#     "WI_geopackage_wetlands.gpkg",
#     "NY_geopackage_wetlands.gpkg",
#     "VA_geopackage_wetlands.gpkg",
#     "TN_geopackage_wetlands.gpkg",
#     "MD_geopackage_wetlands.gpkg",
#     "NJ_geopackage_wetlands.gpkg",
#     "DE_geopackage_wetlands.gpkg",
#     "MO_geopackage_wetlands.gpkg",
# ]

# Every Palustrine type: forest, scrub-shrub, emergent marsh AND the ponds (PUB) and
# aquatic beds (PAB) the old filter dropped. Those small isolated ponds are the
# fishless breeding habitat Ambystoma depends on, so excluding them measured the
# distance to the nearest vegetated marsh - which is exactly what a breeding pond
# is not.
WETLAND_WHERE = "ATTRIBUTE LIKE 'P%'"


# 2. Extract Master Grid Metadata & EXACT CRS
with rasterio.open(master_raster_path) as src:
    master_meta      = src.meta.copy()
    master_transform = src.transform
    master_shape     = src.shape  # (44027, 45094)
    master_crs       = src.crs    # Use the EXACT CRS from the master raster!
    master_bounds    = tuple(src.bounds)

log(f"Master Grid loaded. Shape: {master_shape}")

mask_meta = master_meta.copy()
mask_meta.update(
    dtype=rasterio.uint8,
    nodata=0,
    count=1,
    compress="lzw",
    tiled=True,       # window writes stay cheap on a tiled GeoTIFF
    blockxsize=512,
    blockysize=512,
)


# 3. Burn every study state - plus the neighbours, for context only - into one
# shared mask, one spatial tile at a time.
# No full-extent array is ever held in RAM: each tile is rasterized into just
# its own window and written back, preserving pixels burned earlier.
# "w+" (not "w") because every tile reads its window back before overwriting it.
# Tiles are cut from each layer's own footprint on the grid, so a state costs
# one pass over its extent instead of one pass per 20k polygons.
# Resolve every layer up front. A package that does not match fails here in
# seconds instead of after hours of burning - which is exactly what happened
# when IL turned out to hold a Historic_Map_Info table where the wetlands layer
# was expected, six study states into the run.
log("Stage 3/5: burning NWI wetlands into the shared mask")
layer_choice = {}
layer_features = {}
for file in state_files + context_files:
    file_path = os.path.join(nwi_folder, file)
    if os.path.exists(file_path):
        layer_choice[file] = find_wetland_layer(file_path, file.split("_")[0])
        # How big this job is, known before spending hours on it. Read from the
        # package's bookkeeping tables - see layer_meta for why not read_info.
        layer_features[file] = layer_meta(
            file_path, layer_choice[file])["features"]
        # The burn fetches one tile at a time, which is only cheap with an RTree
        # index. NWI's downloads have none and /kaggle/input is read-only, so
        # those packages get an indexed working copy when their turn comes.
        indexed = has_spatial_index(file_path, layer_choice[file])
        log(f"  {file}: layer {layer_choice[file]} | "
            f"{count_str(layer_features[file])} features"
            f"{'' if indexed else ' | no RTree, will index a copy'}")
    else:
        log(f"  {file}: MISSING")

state_timings = []

with rasterio.open(temp_mask_raster, "w+", **mask_meta) as dest:

    missing_context = []

    for file, is_context in ([(f, False) for f in state_files]
                             + [(f, True) for f in context_files]):

        file_path = os.path.join(nwi_folder, file)

        if not os.path.exists(file_path):
            if is_context:
                missing_context.append(file.split("_")[0])
            else:
                log(f"WARNING: {file} not found - that study state is missing entirely")
            continue

        target_layer = layer_choice[file]
        target_path = file_path
        if not has_spatial_index(file_path, target_layer):
            target_path = indexed_copy(file_path, target_layer)
        meta = layer_meta(target_path, target_layer)
        layer_crs = meta["crs"]
        layer_bounds = meta["bounds"]

        if layer_crs is None:
            raise RuntimeError(f"{file}: layer {target_layer} has no CRS, cannot tile it")

        log(f"Processing {file} ({'context' if is_context else 'study'}, "
            f"layer {target_layer})...")

        burned = 0
        tiles = 0
        state_start = time.perf_counter()
        t_read = t_reproj = t_burn = 0.0

        # Tile this layer's own footprint on the master grid, in pixel-aligned
        # windows: each tile is fetched once through the spatial index and written
        # back once, so disk I/O is the state's footprint - not one pass per
        # row-numbered chunk over the state's whole bounding box.
        if layer_bounds is None:
            # No RTree to take the extent from. Burn the whole grid rather than
            # guess: an empty window here would quietly skip the state.
            log(f"  {file}: extent unknown, tiling the whole grid")
            state_win = Window(0, 0, master_shape[1], master_shape[0])
        else:
            sb = transform_bounds(layer_crs, master_crs, *layer_bounds, densify_pts=21)
            state_win = pixel_window(
                (max(sb[0], master_bounds[0]), max(sb[1], master_bounds[1]),
                 min(sb[2], master_bounds[2]), min(sb[3], master_bounds[3])),
                master_transform, master_shape)

        if state_win.width <= 0 or state_win.height <= 0:
            log(f"  -> {file}: nothing on the master grid")
            state_win = Window(0, 0, 0, 0)  # the tile loops below then no-op

        col_stop = int(state_win.col_off + state_win.width)
        row_stop = int(state_win.row_off + state_win.height)

        for row_off in range(int(state_win.row_off), row_stop, TILE_PX):
            for col_off in range(int(state_win.col_off), col_stop, TILE_PX):

                win = Window(col_off, row_off,
                             min(TILE_PX, col_stop - col_off),
                             min(TILE_PX, row_stop - row_off))

                # pyogrio does not reproject bbox: express the tile in layer CRS.
                # Padded by one master pixel: the RTree keeps its boxes as 32-bit
                # floats, so a feature whose envelope rounds inside by a fraction
                # of a pixel could otherwise be dropped at a tile seam. A feature
                # that does not touch the tile contributes no pixels either way.
                tb = bounds(win, master_transform)
                pad = max(abs(master_transform.a), abs(master_transform.e))
                tile_bbox = transform_bounds(master_crs, layer_crs,
                                             tb[0] - pad, tb[1] - pad,
                                             tb[2] + pad, tb[3] + pad,
                                             densify_pts=21)

                chunk_start = time.perf_counter()
                gdf = gpd.read_file(
                    target_path,
                    layer=target_layer,
                    engine="pyogrio",
                    where=WETLAND_WHERE,
                    bbox=tile_bbox,
                    columns=[],  # Only load geometries, skip attribute columns to save RAM
                )
                chunk_read = time.perf_counter() - chunk_start

                if len(gdf) == 0:
                    continue  # empty tile, not the end of the state

                tiles += 1
                gdf = gdf[gdf.geometry.notna()]

                if gdf.empty:
                    del gdf
                    gc.collect()
                    continue

                chunk_reproj = 0.0
                if gdf.crs != master_crs:
                    reproj_start = time.perf_counter()
                    gdf = gdf.to_crs(master_crs)
                    chunk_reproj = time.perf_counter() - reproj_start

                # Read-modify-write, so pixels burned by earlier states survive.
                burn_start = time.perf_counter()
                tile = dest.read(1, window=win)
                features.rasterize(
                    shapes=((geom, 1) for geom in gdf.geometry if geom is not None),
                    out=tile,
                    transform=window_transform(win, master_transform),
                    merge_alg=rasterio.enums.MergeAlg.replace,
                    dtype=np.uint8,
                    all_touched=True
                )
                dest.write(tile, 1, window=win)
                del tile
                chunk_burn = time.perf_counter() - burn_start

                n_burned = len(gdf)
                burned += n_burned
                del gdf
                gc.collect()

                t_read += chunk_read
                t_reproj += chunk_reproj
                t_burn += chunk_burn

                # A feature straddling a tile edge is read by both tiles, so
                # `burned` can run a little ahead of the layer's feature count.
                log(f"    tile {tiles:>3} ({row_off:>6},{col_off:>6}) "
                    f"| read {chunk_read:5.1f}s reproj {chunk_reproj:5.1f}s "
                    f"burn {chunk_burn:5.1f}s | {n_burned} features")

        state_seconds = time.perf_counter() - state_start
        state_timings.append(dict(
            file=file, kind="context" if is_context else "study",
            features=layer_features.get(file), burned=burned, tiles=tiles,
            read=t_read, reproj=t_reproj, burn=t_burn, total=state_seconds,
        ))
        log(f"  -> Burned {burned:,} of {count_str(layer_features.get(file))} features "
            f"in {tiles} tiles | read {fmt(t_read)} reproj {fmt(t_reproj)} "
            f"burn {fmt(t_burn)} | total {fmt(state_seconds)} | "
            f"peak RAM {peak_rss_mib():.0f} MiB")

        if target_path != file_path:
            # Drop the copy before the next state, so peak disk stays at one
            # package instead of the whole set.
            os.remove(target_path)

if missing_context:
    log(f"WARNING: no NWI data for {', '.join(sorted(missing_context))} - wetland "
        f"distance stays inflated along those stretches of the state line.")
log(f"Neighbour context: {len(context_files) - len(missing_context)}"
    f"/{len(context_files)} states burned")

# Where the burn time went, per package. `read` is the paging/IO cost, `burn` is
# the rasterize plus the window read/write, `total` is what that state really
# costs. This is the line that says which package eats the hours.
log(f"Burn summary | {'package':<26}{'kind':>8}{'features':>10}{'tiles':>7}"
    f"{'read':>9}{'reproj':>9}{'burn':>9}{'total':>9}")
for row in state_timings:
    log(f"             | {row['file']:<26}{row['kind']:>8}"
        f"{count_str(row['features']):>10}"
        f"{row['tiles']:>7}{fmt(row['read']):>9}{fmt(row['reproj']):>9}"
        f"{fmt(row['burn']):>9}{fmt(row['total']):>9}")
log(f"             | {'TOTAL':<26}{'':>8}"
    f"{count_str(sum(r['features'] or 0 for r in state_timings)):>10}"
    f"{sum(r['tiles'] for r in state_timings):>7}"
    f"{fmt(sum(r['read'] for r in state_timings)):>9}"
    f"{fmt(sum(r['reproj'] for r in state_timings)):>9}"
    f"{fmt(sum(r['burn'] for r in state_timings)):>9}"
    f"{fmt(sum(r['total'] for r in state_timings)):>9}")

log(f"Mask written: {temp_mask_raster} "
    f"({os.path.getsize(temp_mask_raster) / 2**20:.0f} MiB on disk)")
log(f"Peak RAM after mask write: {peak_rss_mib():.0f} MiB")


# 4. Calculate Euclidean Distance Out-of-Core via GDAL
log("Stage 4/5: calculating continuous distance in meters using gdal_proximity...")
proximity_start = time.perf_counter()

# -values 1:      measure distance to pixels with value 1 (wetlands)
# -distunits GEO: automatically multiplies pixel distance by 30m (georeferenced units)
# -ot Float32:    keeps output in 32-bit float instead of 64-bit float
cmd = [
    "gdal_proximity.py",
    temp_mask_raster,
    output_distance_raster,
    "-values", "1",
    "-distunits", "GEO",
    "-ot", "Float32",
    "-co", "COMPRESS=LZW",
    "-co", "BIGTIFF=YES",
]

subprocess.run(cmd, check=True)
log(f"gdal_proximity finished in {fmt(time.perf_counter() - proximity_start)} | "
    f"peak RAM {peak_rss_mib():.0f} MiB")


# 5. Clip to the 6-state cutline AND drop open water, which is the land mask the
# TWI layer uses. Annual NLCD class 11 was previously left in, so Great Lakes state
# waters (inside the cutline, so not nodata) kept distances of 10-90 km that no land
# pixel ever sees; those pixels are now nodata here exactly as they are in the TWI
# raster.
log("Stage 5/5: applying 6-state cutline mask from master NLCD raster...")
clip_start = time.perf_counter()

DST_NODATA = -9999.0

with rasterio.open(master_raster_path) as master_src, \
     rasterio.open(output_distance_raster, "r+") as dist_dst:

    master_nodata = master_src.nodata
    if master_nodata is None:
        master_nodata = MASTER_NODATA
    dist_dst.nodata = DST_NODATA

    kept = dropped_water = dropped_outside = 0
    max_kept = 0.0

    # Stream in 2048-row strips so RAM stays under ~400 MiB
    step = 2048
    for row_off in range(0, master_src.height, step):
        h = min(step, master_src.height - row_off)
        win = Window(0, row_off, master_src.width, h)

        master_chunk = master_src.read(1, window=win)
        dist_chunk = dist_dst.read(1, window=win)

        # nodata = outside the six states; class 11 = open water (Great Lakes, rivers,
        # reservoirs). Neither carries a meaningful distance to wetland.
        outside = master_chunk == master_nodata
        water = master_chunk == NLCD_OPEN_WATER
        keep = ~(outside | water)

        kept += int(keep.sum())
        dropped_water += int((water & ~outside).sum())
        dropped_outside += int(outside.sum())

        if keep.any():
            max_kept = max(max_kept, float(dist_chunk[keep].max()))

        dist_chunk[~keep] = DST_NODATA
        dist_dst.write(dist_chunk, 1, window=win)

log(f"wetland_distance.tiff clipped to the 6-state cutline and lakes: "
    f"kept {kept/1e6:.1f} M px | open water: {dropped_water/1e6:.1f} M px | "
    f"outside the cutline: {dropped_outside/1e6:.1f} M px | "
    f"max on land {max_kept/1000:.1f} km")
log(f"cutline clip finished in {fmt(time.perf_counter() - clip_start)}")

# Clean up temporary mask file
os.remove(temp_mask_raster)

log(f"Done! Saved aligned distance raster to {output_distance_raster}")


In [ ]:
from rasterio.enums import Resampling
from rasterio.windows import Window
import matplotlib.pyplot as plt
import numpy as np
import rasterio


path = '/kaggle/working/wetland_distance.tiff'

with rasterio.open(path) as src:
    print(f"{src.width} x {src.height} px @ {abs(src.transform.a):.0f} m | "
          f"CRS {src.crs} | nodata {src.nodata}")

    # Overview. At this size one plotted pixel averages ~57x57 source pixels
    # (~1.7 km), so individual wetlands cannot survive this view.
    max_size = 1500
    scale = min(max_size / src.width, max_size / src.height)
    width = max(1, int(src.width * scale))
    height = max(1, int(src.height * scale))

    image = src.read(
        1,
        out_shape=(height, width),
        resampling=Resampling.average,
        masked=True,
    )

    values = image.filled(np.nan).astype('float64')
    valid = values[np.isfinite(values)]
    cap = max(float(np.percentile(valid, 95)), 1000.0)
    print("distance (m)  min {:.0f} | p50 {:.0f} | p90 {:.0f} | p95 {:.0f} | "
          "p99 {:.0f} | max {:.0f}".format(
              valid.min(), *np.percentile(valid, [50, 90, 95, 99]), valid.max()))
    print(f"overview pixels within 1 km of a wetland: {(valid < 1000).mean():.1%}")

    # Same check TWI_raster_prep does, now covering the neighbour states too: no
    # valid distance may survive on open water or outside the six states. The second
    # is what proves the IL/WI/NY/... context polygons were masked back out instead
    # of leaking into the covariate.
    with rasterio.open(master_raster_path) as master_src:
        lc_chk = master_src.read(1, out_shape=(height, width),
                                 resampling=Resampling.nearest)
    dist_chk = src.read(1, out_shape=(height, width),
                        resampling=Resampling.nearest)
    has_value = dist_chk != src.nodata
    leaked_water = int(((lc_chk == NLCD_OPEN_WATER) & has_value).sum())
    leaked_out = int(((lc_chk == MASTER_NODATA) & has_value).sum())
    print(f"mask check  : {leaked_water} water and {leaked_out} outside-cutline valid "
          f"pixels of {int(has_value.sum()):,} sampled")
    if leaked_water or leaked_out:
        raise RuntimeError("wetland distance has valid values outside the land mask")

    # Full-resolution 90 km look at the wetland-densest part of the overview.
    row, col = np.unravel_index(np.nanargmin(values), values.shape)
    src_row = int(row * src.height / height)
    src_col = int(col * src.width / width)
    half = 1500  # 1500 px * 30 m = 45 km
    row_off = min(max(0, src_row - half), max(0, src.height - 2 * half))
    col_off = min(max(0, src_col - half), max(0, src.width - 2 * half))

    # masked: lakes and the area outside the cutline are nodata now, and an
    # unmasked -9999 would plot as dark purple (i.e. as if it were 0 m).
    crop = src.read(1, window=Window(col_off, row_off, 2 * half, 2 * half),
                    masked=True)


fig, (full, zoom) = plt.subplots(1, 2, figsize=(16, 8))

# Capped near the 95th percentile. Left uncapped, the few-hundred-km values
# in the empty corners become vmax and every real distance collapses to purple.
im = full.imshow(image, cmap='viridis', vmax=cap)
full.set_title(f'Distance to nearest wetland (capped at {cap / 1000:.0f} km)')
full.axis('off')
fig.colorbar(im, ax=full, shrink=0.7, label='m')

im = zoom.imshow(crop, cmap='viridis', vmax=2000)
zoom.set_title(f'90 km at full 30 m (row {row_off}, col {col_off})')
zoom.axis('off')
fig.colorbar(im, ax=zoom, shrink=0.7, label='m')

plt.tight_layout()
plt.show()
